# Semantic Correspondence — Local (end-to-end)

Same pipeline as Colab: dataset + weights (if missing), `config.yaml`, `run_pipeline.py`, analysis. Run from repo root with `pip install -e ".[notebook]"` and a CUDA-capable PyTorch wheel.

**Resume:** re-run cells; `pipeline_resume` + `*_resume.pt` (interval 500 by default).

### Device

In [ ]:
import os
import sys
from pathlib import Path

import torch

REPO_DIR = Path.cwd().resolve()
os.chdir(REPO_DIR)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Platform:", sys.platform)

if torch.cuda.is_available():
    DEVICE = "cuda"
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024**3)
    cc = f"{props.major}.{props.minor}"
    print(f"CUDA: {torch.cuda.get_device_name(0)}  VRAM_GB={vram_gb:.1f}  cap={cc}")
    BF16_CUDA = torch.cuda.is_bf16_supported() and props.major >= 8
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    BF16_CUDA = False
    print("MPS: training uses fp32; batch sizes like small CUDA.")
else:
    DEVICE = "cpu"
    BF16_CUDA = False
    print("CPU: slow; consider reducing epochs in config cell.")

print("DEVICE =", DEVICE)


### Install package (once per environment)

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=str(REPO_DIR),
    check=True,
)
print("pip install OK")


### Throughput (CUDA)

In [ ]:
import torch

if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except (RuntimeError, AttributeError):
        pass
    print("cuDNN benchmark + TF32 on")
else:
    try:
        torch.set_float32_matmul_precision("high")
    except (RuntimeError, AttributeError):
        pass


### SPair-71k

Download/extract into `data/SPair-71k/` if missing.

In [ ]:
import os
import sys
import tarfile
import urllib.request
from pathlib import Path

SPAIR_URL = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
DATA_DIR = REPO_DIR / "data"
SPAIR_ROOT = DATA_DIR / "SPair-71k"
TAR_PATH = DATA_DIR / "SPair-71k.tar.gz"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not SPAIR_ROOT.is_dir():
    if not TAR_PATH.is_file():
        print("Downloading:", SPAIR_URL)
        urllib.request.urlretrieve(SPAIR_URL, TAR_PATH)
    print("Extracting to:", DATA_DIR)
    with tarfile.open(TAR_PATH, "r:gz") as tar:
        if sys.version_info >= (3, 12):
            tar.extractall(path=DATA_DIR, filter="data")
        else:
            tar.extractall(path=DATA_DIR)
else:
    print("Dataset present:", SPAIR_ROOT)

print("SPAIR_ROOT exists:", SPAIR_ROOT.is_dir())


### Pretrained weights

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/download_pretrained_weights.py"], cwd=str(REPO_DIR), check=True)


### Config

| VRAM / GPU | FT DINO | FT SAM | LoRA DINO | LoRA SAM | precision |
|------------|---------|--------|-----------|----------|-----------|
| ~11 GB (e.g. 1080 Ti) | 25 | 10 | 25 | 10 | fp16 (see below) |
| ~24–40 GB | 20–32 | 4–8 | 32–48 | 8 | auto |
| ~80 GB (H100) | 32 | 8 | 48 | 8 | auto |
| ~32 GB (M2 Max unified, MPS) | 8 | 3 | 12 | 3 | fp32 (MPS forces fp32) |
| ~16 GB (M1/M2, MPS) | 4 | 2 | 4 | 2 | fp32 |

**Throughput:** If the GPU is pegged at ~100% but VRAM stays low, increase the batch maps in the next cell until memory is ~70–85% or you hit OOM, then step back one notch.

**Data loading:** Default **`num_workers = 1`** on CUDA. Each extra worker is a separate process: together they can **RAM-multiply** decoding/buffers and push the machine into **swap** → wall time explodes (e.g. tens of minutes per few dozen batches). Raise to `2`–`4` only if RAM stays comfortable and `top`/`Activity Monitor` shows little swap. Avoid **`-1`** on desktops.

**Resume vs cold start:** The next cell sets `START_FROM_SCRATCH` automatically. If both `runs/` and `checkpoints/` exist under the repo root, the pipeline **resumes** (skip completed stages, `*_resume.pt` for training). If you delete **either** folder, the next run is a **cold start** (full pipeline, no stage skip, no training resume). Re-run the Config cell after deleting folders.

**Tuning `num_workers`:** Changing only workers does **not** require deleting checkpoints. `num_workers` is not part of the pipeline config fingerprint, so stage skip stays valid; training still loads `*_resume.pt` and continues. It affects **wall time** and **RAM/swap**, not the loss curve given the same batch size and LR (same optimization steps).

**Apple Silicon (MPS):** precision is forced to fp32 (MPS does not support bf16/fp16 reliably); `num_workers` stays at 1 because unified memory makes extra workers RAM-multiplying. Batch sizes above assume the default image sizes (DINOv2 518, DINOv3/SAM 512).


In [ ]:
import yaml
from pathlib import Path

RUNS_DIR = REPO_DIR / "runs"
CKPT_DIR = REPO_DIR / "checkpoints"
# Both dirs must exist to resume (stage skip + *_resume.pt). Delete either folder for a cold start.
_has_resume_context = RUNS_DIR.is_dir() and CKPT_DIR.is_dir()
START_FROM_SCRATCH = not _has_resume_context

RUNTIME_DEVICE = DEVICE
if DEVICE == "cuda" and not BF16_CUDA:
    PRECISION = "fp16"
else:
    PRECISION = "auto"

# CUDA: 1 worker minimizes RAM and swap thrashing on typical desktops (often faster wall-clock than 4-8).
NUM_WORKERS = 1
FT_EPOCHS = 50
FT_PATIENCE = 7
LORA_EPOCHS = 50
LORA_PATIENCE = 7

if DEVICE == "cuda" and BF16_CUDA:
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 32, "dinov3_vitb16": 32, "sam_vit_b": 8}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 48, "dinov3_vitb16": 48, "sam_vit_b": 8}
    FT_FALLBACK = 32
    LORA_FALLBACK = 48
elif DEVICE == "cuda":
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 25, "dinov3_vitb16": 25, "sam_vit_b": 10}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 25, "dinov3_vitb16": 25, "sam_vit_b": 10}
    FT_FALLBACK = 25
    LORA_FALLBACK = 25
elif DEVICE == "mps":
    # Apple Silicon unified memory (M1/M2/M3). Tuned for ~32 GB (e.g. M2 Max).
    # fp32 is forced by the precision resolver; num_workers=1 keeps RAM pressure low.
    # On a 16 GB Mac halve these (DINO=4, SAM=2).
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 8, "dinov3_vitb16": 8, "sam_vit_b": 3}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 12, "dinov3_vitb16": 12, "sam_vit_b": 3}
    FT_FALLBACK = 8
    LORA_FALLBACK = 12
else:
    # CPU (last-resort). Keep minimal.
    FT_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 4, "dinov3_vitb16": 4, "sam_vit_b": 2}
    LORA_BATCH_SIZE_BY_BACKBONE = {"dinov2_vitb14": 4, "dinov3_vitb16": 4, "sam_vit_b": 2}
    FT_FALLBACK = 4
    LORA_FALLBACK = 4

FT_LR = 5e-5
FT_WEIGHT_DECAY = 0.01
LORA_LR = 1e-3
LORA_ALPHA = 16.0
LORA_RANK = 8
LORA_LAST_BLOCKS = 2
LAST_BLOCKS_LIST = [1, 2, 4]
RUN_PYTEST = False
IMAGE_SIZE_BY_BACKBONE = {
    "dinov2_vitb14": [518, 518],
    "dinov3_vitb16": [512, 512],
    "sam_vit_b": [512, 512],
}
LOG_BATCH_INTERVAL = 50
RESUME_SAVE_INTERVAL = 500

cfg = {
    "paths": {
        "spair_root": str(REPO_DIR / "data" / "SPair-71k"),
        "checkpoint_dir": str(CKPT_DIR),
    },
    "runtime": {
        "device": RUNTIME_DEVICE,
        "precision": PRECISION,
        "num_workers": NUM_WORKERS,
        "preprocess": "FIXED_RESIZE",
        "image_size_by_backbone": IMAGE_SIZE_BY_BACKBONE,
        "limit_pairs": 0,
        "eval_split": "test",
        "alphas": [0.05, 0.1, 0.2],
        "wsa_window": 5,
        "wsa_temperature": 1.0,
        "log_batch_interval": LOG_BATCH_INTERVAL,
        "resume_save_interval": RESUME_SAVE_INTERVAL,
    },
    "finetune": {
        "last_blocks": LAST_BLOCKS_LIST,
        "epochs": FT_EPOCHS,
        "patience": FT_PATIENCE,
        "batch_size": FT_FALLBACK,
        "batch_size_by_backbone": FT_BATCH_SIZE_BY_BACKBONE,
        "lr": FT_LR,
        "weight_decay": FT_WEIGHT_DECAY,
        "dinov2_weights": str(CKPT_DIR / "dinov2_vitb14_pretrain.pth"),
        "dinov3_weights": str(CKPT_DIR / "dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
        "sam_checkpoint": str(CKPT_DIR / "sam_vit_b_01ec64.pth"),
    },
    "lora": {
        "rank": LORA_RANK,
        "alpha": LORA_ALPHA,
        "lr": LORA_LR,
        "last_blocks": LORA_LAST_BLOCKS,
        "epochs": LORA_EPOCHS,
        "patience": LORA_PATIENCE,
        "batch_size": LORA_FALLBACK,
        "batch_size_by_backbone": LORA_BATCH_SIZE_BY_BACKBONE,
    },
    "workflow_toggles": {
        "run_verify_dataset": True,
        "train_finetune": [True, True, True],
        "train_lora": [True, True, True],
        "run_eval_baseline": [True, True, True],
        "run_eval_baseline_wsa": [True, True, True],
        "run_eval_finetuned_checkpoint": [True, True, True],
        "run_eval_lora_checkpoint": [True, True, True],
        "run_eval_finetuned_wsa": [True, True, True],
        "run_eval_lora_wsa": [True, True, True],
        "run_export_metrics_tables": True,
        "run_pytest": RUN_PYTEST,
        "pipeline_resume": not START_FROM_SCRATCH,
    },
}

cfg_path = REPO_DIR / "config.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

print("Written:", cfg_path)
print("device:", RUNTIME_DEVICE, "precision:", PRECISION, "workers:", NUM_WORKERS)
print(
    "Pipeline:",
    "resume" if _has_resume_context else "cold start",
    "| START_FROM_SCRATCH =",
    START_FROM_SCRATCH,
)
print("FT batches:", FT_BATCH_SIZE_BY_BACKBONE, "LoRA:", LORA_BATCH_SIZE_BY_BACKBONE)


### Run pipeline

In [ ]:
import json
import os
import subprocess
import sys
import threading
from collections import deque
from pathlib import Path

from IPython.display import clear_output

os.chdir(REPO_DIR)

env = dict(os.environ)
if START_FROM_SCRATCH:
    env["SEMANTIC_CORRESPONDENCE_PIPELINE_RESET"] = "1"
else:
    env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_RESET", None)
env.pop("SEMANTIC_CORRESPONDENCE_PIPELINE_LOG_FILE_ONLY", None)
env["PYTHONUNBUFFERED"] = "1"
# MPS: some ops (e.g. grid_sampler_2d_backward) have no MPS kernel yet; fall back to CPU.
if DEVICE == "mps":
    env["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
_pp = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = f"{REPO_DIR}{os.pathsep}{_pp}" if _pp else str(REPO_DIR)

STAGE_EVENTS_PATH = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
TAIL_LINES = 40
REFRESH_SECONDS = 3.0
cmd = [sys.executable, "-u", "scripts/run_pipeline.py", "--config", "config.yaml"]

_output_lines: deque = deque(maxlen=TAIL_LINES)
_proc_done = threading.Event()
_return_code: list = []


def _stream_subprocess() -> None:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
        cwd=str(REPO_DIR),
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        _output_lines.append(line.rstrip())
    _return_code.append(proc.wait())
    _proc_done.set()


threading.Thread(target=_stream_subprocess, daemon=True).start()


def _read_stage_events() -> list:
    if not STAGE_EVENTS_PATH.is_file():
        return []
    events = []
    with open(STAGE_EVENTS_PATH, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                events.append(json.loads(raw))
            except json.JSONDecodeError:
                pass
    return events


def _render(events: list) -> None:
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    latest_start = next((e for e in reversed(events) if e.get("action") == "start"), None)
    current = latest_start["stage_id"] if latest_start else "(waiting...)"
    rc = _return_code[0] if _return_code else None
    if not _proc_done.is_set():
        status = "RUNNING"
    elif rc == 0:
        status = "COMPLETED"
    else:
        status = f"FAILED (exit={rc})"
    sep = "=" * 64
    print(sep)
    print(f"  Pipeline: {status}")
    print(f"  Current stage: {current}")
    print(f"  Completed: {len(done):3d}  Skipped: {len(skipped):3d}  Failed: {len(failed):3d}")
    if done:
        tail = done[-6:]
        suffix = "..." if len(done) > 6 else ""
        print(f"  Last done: {', '.join(tail)}{suffix}")
    if failed:
        print(f"  FAILED: {', '.join(failed)}")
    print(sep)
    print(f"--- last {TAIL_LINES} log lines ---")
    for line in _output_lines:
        print(line)


while not _proc_done.is_set():
    clear_output(wait=True)
    _render(_read_stage_events())
    _proc_done.wait(timeout=REFRESH_SECONDS)

clear_output(wait=True)
_render(_read_stage_events())

rc = _return_code[0] if _return_code else -1
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
print("\nPipeline completed successfully.")


### Stage summary

In [ ]:
import json
from pathlib import Path

import pandas as pd

p = REPO_DIR / "runs" / "logs" / "stage_events.jsonl"
if not p.is_file():
    print("No file:", p)
else:
    events = []
    with open(p, "r", encoding="utf-8") as f:
        for raw in f:
            raw = raw.strip()
            if raw:
                try:
                    events.append(json.loads(raw))
                except json.JSONDecodeError:
                    pass
    done = [e["stage_id"] for e in events if e.get("action") == "done"]
    skipped = [e["stage_id"] for e in events if e.get("action") == "skip"]
    failed = [e["stage_id"] for e in events if e.get("action") == "fail"]
    print(f"events={len(events)} done={len(done)} skipped={len(skipped)} failed={len(failed)}")
    if failed:
        print("Failed:", failed)
    display(pd.DataFrame(events).tail(20))


### Training curves

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

CKPT_DIR = REPO_DIR / "checkpoints"
files = sorted(CKPT_DIR.glob("*_history.jsonl"))
if not files:
    print("No history in", CKPT_DIR)
else:
    histories = {}
    for hf in files:
        name = hf.stem.replace("_history", "")
        recs = []
        with open(hf, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    recs.append(json.loads(line))
        if recs:
            histories[name] = recs
    if histories:
        n = len(histories)
        fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
        for col, (name, records) in enumerate(sorted(histories.items())):
            ax = axes[0, col]
            ep = [r["epoch"] for r in records]
            ax.plot(ep, [r["train_loss"] for r in records], marker=".", label="train_loss")
            ax.plot(ep, [r["val_loss"] for r in records], marker=".", label="val_loss")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.set_title(name, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3)
        fig.suptitle("Training curves", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("Empty history files.")


## Results

### Load exports

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

os.chdir(REPO_DIR)
EXPORTS = REPO_DIR / "runs" / "pipeline_exports"


def _load_json(name):
    p = EXPORTS / name
    if p.is_file():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    print("Not found:", p)
    return None


pck_results = _load_json("pck_results.json")
per_category = _load_json("pck_results_per_category.json")
by_difficulty = _load_json("pck_results_by_difficulty_flag.json")
print("Loaded", len(pck_results or []), "eval runs.")


### Aggregate PCK

In [ ]:
if pck_results:
    rows = []
    for r in pck_results:
        row = {"name": r["name"]}
        row.update(r.get("metrics", {}))
        rows.append(row)
    df_pck = pd.DataFrame(rows)
    display(
        df_pck.style.format({c: "{:.4f}" for c in df_pck.columns if c.startswith("pck@")}).set_caption(
            "PCK"
        )
    )
else:
    print("No results.")


### Fine-tune depth

In [ ]:
import re

import matplotlib.pyplot as plt

if pck_results:
    ft_rows = []
    for r in pck_results:
        m = re.match(r"(.+)_ft_lb(\d+)$", r["name"])
        if m:
            row = {"backbone": m.group(1), "last_blocks": int(m.group(2))}
            row.update(r.get("metrics", {}))
            ft_rows.append(row)
    if ft_rows:
        df_ft = pd.DataFrame(ft_rows).sort_values(["backbone", "last_blocks"])
        display(df_ft)
        fig, axes = plt.subplots(
            1, len(df_ft["backbone"].unique()), figsize=(5 * len(df_ft["backbone"].unique()), 4), squeeze=False
        )
        for col_idx, (bb, grp) in enumerate(df_ft.groupby("backbone")):
            ax = axes[0, col_idx]
            for mc in [c for c in grp.columns if c.startswith("pck@")]:
                ax.plot(grp["last_blocks"], grp[mc], marker="o", label=mc)
            ax.set_xlabel("Unfrozen blocks")
            ax.set_ylabel("PCK")
            ax.set_title(bb)
            ax.legend(fontsize=8)
            ax.set_xticks(sorted(grp["last_blocks"].unique()))
        fig.suptitle("PCK vs fine-tuned blocks", fontsize=13)
        fig.tight_layout()
        plt.show()
    else:
        print("No *_ft_lb<N> rows.")
else:
    print("No pck_results.")


### Per-category PCK

In [ ]:
import numpy as np

if per_category:
    rows = []
    for entry in per_category:
        for cat, alphas in entry.get("categories", {}).items():
            row = {"run": entry["name"], "category": cat}
            row.update(alphas)
            rows.append(row)
    if rows:
        df_cat = pd.DataFrame(rows)
        pck_col = [c for c in df_cat.columns if c.startswith("pck@")]
        if pck_col:
            pivot = df_cat.pivot_table(index="category", columns="run", values=pck_col[0], aggfunc="first")
            fig, ax = plt.subplots(figsize=(max(12, len(pivot.columns) * 1.5), max(6, len(pivot) * 0.4)))
            im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index, fontsize=8)
            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    v = pivot.values[i, j]
                    if not np.isnan(v):
                        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7)
            fig.colorbar(im, ax=ax, label=pck_col[0])
            ax.set_title(f"Per-category {pck_col[0]}")
            fig.tight_layout()
            plt.show()
    else:
        print("No per-category rows.")
else:
    print("No per_category export.")


### Per-difficulty

In [ ]:
if by_difficulty:
    rows = []
    for entry in by_difficulty:
        run_name = entry["name"]
        for flag, buckets in entry.get("data", {}).items():
            for bucket, info in buckets.items():
                summary = info.get("summary", {}).get("image", {})
                for metric_key, val_dict in summary.items():
                    rows.append(
                        {
                            "run": run_name,
                            "flag": flag,
                            "value": int(bucket),
                            "metric": metric_key.replace("custom_", ""),
                            "pck": val_dict.get("all", float("nan")),
                        }
                    )
    if rows:
        import pandas as pd

        df_diff = pd.DataFrame(rows)
        display(df_diff.pivot_table(index=["run", "metric"], columns=["flag", "value"], values="pck").round(4))
    else:
        print("No difficulty rows.")
else:
    print("No difficulty export.")


### Qualitative (DINOv2 baseline)

In [ ]:
import matplotlib.pyplot as plt

import torch
from data.dataset import PreprocessMode, SPair71kPairDataset
from evaluation.visualize import visualize_correspondences
from models.common import (
    BackboneName,
    DenseExtractorConfig,
    DenseFeatureExtractor,
    predict_correspondences_cosine_argmax,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

spair_root = str(REPO_DIR / "data" / "SPair-71k")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

ds = SPair71kPairDataset(
    spair_root=spair_root,
    split="test",
    preprocess=PreprocessMode.FIXED_RESIZE,
    output_size_hw=(784, 784),
    normalize=True,
)
cfg_ext = DenseExtractorConfig(
    name=BackboneName.DINOV2_VIT_B14,
    dinov2_weights_path=str(REPO_DIR / "checkpoints" / "dinov2_vitb14_pretrain.pth"),
)
extractor = DenseFeatureExtractor(cfg_ext).to(device).eval()

NUM_EXAMPLES = 4
indices = list(range(0, min(len(ds), NUM_EXAMPLES * 50), max(1, len(ds) // NUM_EXAMPLES)))[:NUM_EXAMPLES]

for idx in indices:
    sample = ds[idx]
    src = sample["src_img"].unsqueeze(0).to(device)
    tgt = sample["tgt_img"].unsqueeze(0).to(device)
    src_kps = sample["src_kps"].to(device)
    tgt_kps = sample["tgt_kps"].to(device)
    pck_thr = float(sample["pck_threshold_bbox"])
    with torch.no_grad():
        feat_src, _ = extractor(src)
        feat_tgt, _ = extractor(tgt)
        out = predict_correspondences_cosine_argmax(
            feat_src,
            feat_tgt,
            src_kps,
            img_hw=(784, 784),
            img_hw_src=(784, 784),
            img_hw_tgt=(784, 784),
        )
        pred_tgt = out["pred_tgt_xy"]
    fig = visualize_correspondences(
        sample["src_img"],
        sample["tgt_img"],
        src_kps,
        pred_tgt,
        tgt_kps,
        pck_threshold=pck_thr,
        alpha=0.1,
        title=f"Pair {idx} (category: {sample.get('category', '?')})",
    )
    plt.show()
    plt.close(fig)
